In [1]:
import sqlite3
from typing import List, Optional

DB_PATH = 'lab3_blockchain.db'

def get_connection():
    conn = sqlite3.connect(DB_PATH)
    conn.execute("PRAGMA foreign_keys = ON")
    conn.row_factory = sqlite3.Row
    return conn

In [2]:
class Block:
    def __init__(self, id: str, view: int, desc: str, img: Optional[bytes] = None):
        self.id = id
        self.view = view
        self.desc = desc
        self.img = img

    @classmethod
    def from_row(cls, row: sqlite3.Row) -> 'Block':
        return cls(
            id=row['id'],
            view=row['view'],
            desc=row['desc'],
            img=row['img']
        )

    @classmethod
    def get_all(cls) -> List['Block']:
        conn = get_connection()
        cur = conn.cursor()
        cur.execute("SELECT id, view, desc, img FROM BLOCKS")
        rows = cur.fetchall()
        conn.close()
        return [cls.from_row(row) for row in rows]

    @classmethod
    def get_by_id(cls, block_id: str) -> Optional['Block']:
        conn = get_connection()
        cur = conn.cursor()
        cur.execute("SELECT id, view, desc, img FROM BLOCKS WHERE id = ?", (block_id,))
        row = cur.fetchone()
        conn.close()
        return cls.from_row(row) if row else None

    def __repr__(self):
        return f"Block(id={self.id}, view={self.view}, desc={self.desc})"

In [3]:
class Source:
    def __init__(self, id: int, ip_addr: str, country_code: str):
        self.id = id
        self.ip_addr = ip_addr
        self.country_code = country_code

    @classmethod
    def from_row(cls, row: sqlite3.Row) -> 'Source':
        return cls(
            id=row['id'],
            ip_addr=row['ip_addr'],
            country_code=row['country_code']
        )

    @classmethod
    def get_all(cls) -> List['Source']:
        conn = get_connection()
        cur = conn.cursor()
        cur.execute("SELECT id, ip_addr, country_code FROM SOURCES")
        rows = cur.fetchall()
        conn.close()
        return [cls.from_row(row) for row in rows]

    @classmethod
    def get_by_id(cls, source_id: int) -> Optional['Source']:
        conn = get_connection()
        cur = conn.cursor()
        cur.execute("SELECT id, ip_addr, country_code FROM SOURCES WHERE id = ?", (source_id,))
        row = cur.fetchone()
        conn.close()
        return cls.from_row(row) if row else None

    def __repr__(self):
        return f"Source(id={self.id}, ip={self.ip_addr}, country={self.country_code})"

In [4]:
class Person:
    def __init__(self, id: int, name: str, addr: str):
        self.id = id
        self.name = name
        self.addr = addr

    @classmethod
    def from_row(cls, row: sqlite3.Row) -> 'Person':
        return cls(
            id=row['id'],
            name=row['name'],
            addr=row['addr']
        )

    @classmethod
    def get_all(cls) -> List['Person']:
        conn = get_connection()
        cur = conn.cursor()
        cur.execute("SELECT id, name, addr FROM PERSONS")
        rows = cur.fetchall()
        conn.close()
        return [cls.from_row(row) for row in rows]

    @classmethod
    def get_by_id(cls, person_id: int) -> Optional['Person']:
        conn = get_connection()
        cur = conn.cursor()
        cur.execute("SELECT id, name, addr FROM PERSONS WHERE id = ?", (person_id,))
        row = cur.fetchone()
        conn.close()
        return cls.from_row(row) if row else None

    def __repr__(self):
        return f"Person(id={self.id}, name={self.name})"

In [5]:
class Vote:
    def __init__(self, block_id: str, voter_id: int, timestamp: str, source_id: int):
        self.block_id = block_id
        self.voter_id = voter_id
        self.timestamp = timestamp
        self.source_id = source_id

    @classmethod
    def from_row(cls, row: sqlite3.Row) -> 'Vote':
        return cls(
            block_id=row['block_id'],
            voter_id=row['voter_id'],
            timestamp=row['timestamp'],
            source_id=row['source_id']
        )

    @classmethod
    def get_all(cls) -> List['Vote']:
        conn = get_connection()
        cur = conn.cursor()
        cur.execute("SELECT block_id, voter_id, timestamp, source_id FROM VOTES")
        rows = cur.fetchall()
        conn.close()
        return [cls.from_row(row) for row in rows]

    @classmethod
    def get_by_block(cls, block_id: str) -> List['Vote']:
        conn = get_connection()
        cur = conn.cursor()
        cur.execute("SELECT block_id, voter_id, timestamp, source_id FROM VOTES WHERE block_id = ?", (block_id,))
        rows = cur.fetchall()
        conn.close()
        return [cls.from_row(row) for row in rows]

    def get_details(self) -> dict:
        conn = get_connection()
        cur = conn.cursor()
        cur.execute("""
            SELECT 
                v.block_id, v.voter_id, v.timestamp, v.source_id,
                b.view AS block_view, b.desc AS block_desc,
                p.name AS voter_name, p.addr AS voter_addr,
                s.ip_addr AS source_ip, s.country_code AS source_country
            FROM VOTES v
            LEFT JOIN BLOCKS b ON v.block_id = b.id
            LEFT JOIN PERSONS p ON v.voter_id = p.id
            LEFT JOIN SOURCES s ON v.source_id = s.id
            WHERE v.block_id = ? AND v.voter_id = ?
        """, (self.block_id, self.voter_id))
        row = cur.fetchone()
        conn.close()
        return dict(row) if row else {}

    def __repr__(self):
        return f"Vote(block={self.block_id}, voter={self.voter_id}, time={self.timestamp})"

In [7]:
print("=== Усі блоки ===")
blocks = Block.get_all()
for b in blocks:
    print(b)

print("\n=== Блок за ID ===")
b = Block.get_by_id('0xf13a5b7c')
print(b)

print("\n=== Усі джерела ===")
sources = Source.get_all()
for s in sources:
    print(s)

print("\n=== Усі особи ===")
persons = Person.get_all()
for p in persons:
    print(p)

print("\n=== Усі голоси ===")
votes = Vote.get_all()
for v in votes:
    print(v)

print("\n=== Голоси за блок '0xe2dd4f6a' ===")
votes_block = Vote.get_by_block('0xe2dd4f6a')
for v in votes_block:
    print(v)

print("\n=== Деталі першого голосу ===")
if votes:
    details = votes[0].get_details()
    for key, val in details.items():
        print(f"{key}: {val}")

=== Усі блоки ===
Block(id=0x1f3a5b7c, view=1, desc=First block)
Block(id=0x8e2d4f6a, view=2, desc=Second block)
Block(id=0x9c0b3d1e, view=3, desc=Third block)

=== Блок за ID ===
None

=== Усі джерела ===
Source(id=1, ip=203.0.113.1, country=UA)
Source(id=2, ip=198.51.100.2, country=PL)
Source(id=3, ip=192.0.2.3, country=DE)
Source(id=4, ip=10.0.0.4, country=US)

=== Усі особи ===
Person(id=1, name=Катерина Слободяник)
Person(id=2, name=Юлія Дмишко)
Person(id=3, name=Світлана Удичка)

=== Усі голоси ===
Vote(block=0x1f3a5b7c, voter=1, time=2025-02-10 12:00:00)
Vote(block=0x1f3a5b7c, voter=2, time=2025-02-10 12:05:00)
Vote(block=0x8e2d4f6a, voter=1, time=2025-02-10 12:10:00)
Vote(block=0x8e2d4f6a, voter=3, time=2025-02-10 12:15:00)
Vote(block=0x9c0b3d1e, voter=2, time=2025-02-10 12:20:00)
Vote(block=0x9c0b3d1e, voter=3, time=2025-02-10 12:25:00)

=== Голоси за блок '0xe2dd4f6a' ===

=== Деталі першого голосу ===
block_id: 0x1f3a5b7c
voter_id: 1
timestamp: 2025-02-10 12:00:00
source_id: